# Workflow: External Tools (Julia)

One of the main goals of VeriGym is to allow integration between a variety of existing tools and frameworks.
In this workflow, we show off the intergration with one such external framework: the `POMDPs.jl` format used to define and solve (PO)MDPs in the `Julia` programming language.

**What do we show here?**
In this workflow, we benchmark a standard RL algorithm against model checked ground-truth results with the following steps:
1. We construct an abstraction for the Gymnasium environment `X` and export it to UMB format.
2. In `Julia`, we import the model, solve it via an existing solver, and export a policy using `PRISM`s action list format.
3. We import the policy into VeriGym, and evaluate the policy on both the abstracted and original model.


## Imports and helpers

In [13]:
# Imports
import gymnasium as gym
import subprocess


# VeriGym
import verigym
from verigym.policy.randomized import RandomizedPolicy
from verigym.environments.exporter import export_to_umb, export_to_drn

# Paths etc.
dir = "./working_directory/"

## 1 - Construct and export abstraction
To start of, we pick an environment that we want to make an abstraction for. To keep things simple, we use the standard `gym` environment *CartPole*, and use a simple box discretization. For more details on how to construct and discretize models, see the other workflows.

In [ ]:
original_model = verigym.GenerativeEnv.from_gymnasium(gym.make("MountainCar-v0"))

abstracted_model = verigym.create_abstraction(
    original_model, 
    bin_edges_per_action_dim=5,
    bin_edges_per_state_dim=5,
    exploration_policy=RandomizedPolicy(original_model),
    num_steps = int(5e5)
)

Simulation time: 6.8463s
Trajectories in dataset: 22448


/home/merlijn/GitRepos/VeriGym/.venv/lib/python3.13/site-packages/numpy/_core/function_base.py:163: RuntimeWarning: invalid value encountered in multiply
  y *= step
/home/merlijn/GitRepos/VeriGym/.venv/lib/python3.13/site-packages/numpy/_core/function_base.py:173: RuntimeWarning: invalid value encountered in add
  y += start


processing in  30.279924154281616
aggregating..
aggregating in 0.006378173828125
Learning Abstraction: 30.2958s


Next, to use any external tool, we need to be able to *write* our constructed model to a file. VeriGym makes use of existing `stormpy` functionality to supports writing to the following formats:
* [Unified Markov Binary (UMB)](https://arxiv.org/abs/2606.17811);
* [Direct Encoding (DRN)](https://www.stormchecker.org/documentation/background/drn.html).

We chose DRN format, in which case exporting looks as follows:

In [ ]:
export_to_umb(abstracted_model, dir + "model.umb")
export_to_drn(abstracted_model, dir + "model.drn")

Write to file ./working_directory/cartpole_model.drn.


## 2 - External Policy Computation

Since our model is now exported into a standard format, we can use external solvers to compute a policy. For this example, we use `MCTS.jl`, which is a `Julia` program that requires the model in `POMDPs.jl` format. VeriGym includes a number of `Julia` scripts that simplify this process. More precisely, this includes code to import and export MDPs in UMB format, and export policies in a format readable to VeriGym. Using this functionality, we can run a simple `Julia` script to compute and export a policy for our model:


```julia
using POMDPs
using MCTS

include("../src/verigym/external/julia/POMDPs_UMB.jl")
import .POMDPs_UMB   # Ours!

dir = "./working_directory/"
input, output = dir * "model.umb", dir * "policy.txt"
model = POMDPs_UMB.read_umb(input; discount=0.95)

solver = MCTSSolver()
planner = solve(solver, model)

policy = POMDPs_UMB.Explicit_policy(planner, model)
POMDPs_UMB.write_policy(output, policy)
println("Policy computation complete!")
```

The code below runs this script, but requires you to have `Julia` installed. If you do not, skip the next codeblock.

In [20]:
subprocess.call(["julia", "./workflow_external_tools_script.jl"])

Policy computation complete!


0